# Job Ad Skill Extraction & Mapping to ESCO Taxonomy
## Step 1: Exploratory Data Analysis

## Setup

In [ ]:
import json
import pandas as pd
import numpy as np
from collections import Counter
import re
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
sns.set_style('whitegrid')

## 1. Job Ads Data Exploration

### Load Data

In [ ]:
# Load job ads
ads = []
with open('ads-50k.json', 'r', encoding='utf-8') as f:
    for line in f:
        ads.append(json.loads(line))

print(f"Total job ads loaded: {len(ads):,}")

### Data Structure

In [ ]:
# Examine ad structure
sample_ad = ads[0]

print("Sample Ad Structure:")
print(f"ID: {sample_ad['id']}")
print(f"Title: {sample_ad['title']}")
print(f"\nAbstract:\n{sample_ad['abstract']}")
print(f"\nMetadata fields: {list(sample_ad['metadata'].keys())}")
print(f"\nContent: {sample_ad['content'][:200]}...")

In [ ]:
# Extract clean text from HTML
def extract_text_from_html(html_content):
    """Extract clean text from HTML content"""
    soup = BeautifulSoup(html_content, 'html.parser')
    return soup.get_text(separator=' ', strip=True)

# Show cleaned content
content_text = extract_text_from_html(sample_ad['content'])
print("Cleaned Content:")
print(content_text[:800])
print("...")

### Statistics

In [ ]:
# Extract all text fields
titles = [ad['title'] for ad in ads]
abstracts = [ad['abstract'] for ad in ads]
contents = [extract_text_from_html(ad['content']) for ad in ads]

# Calculate mean, median, and max lengths for each text field
stats = pd.DataFrame({
    'Field': ['Title', 'Abstract', 'Content'],
    'Avg Length (chars)': [
        np.mean([len(t) for t in titles]),
        np.mean([len(a) for a in abstracts]),
        np.mean([len(c) for c in contents])
    ],
    'Median Length (chars)': [
        np.median([len(t) for t in titles]),
        np.median([len(a) for a in abstracts]),
        np.median([len(c) for c in contents])
    ],
    'Max Length (chars)': [
        max([len(t) for t in titles]),
        max([len(a) for a in abstracts]),
        max([len(c) for c in contents])
    ]
})

print("Text Field Statistics:")
print(stats.to_string(index=False))

### Job Classifications

In [ ]:
# Extract metadata
classifications = [ad['metadata'].get('classification', {}).get('name', 'Unknown') for ad in ads]
work_types = [ad['metadata'].get('workType', {}).get('name', 'Unknown') for ad in ads]

# Top classifications
classification_counts = Counter(classifications)
top_classifications = pd.DataFrame(
    classification_counts.most_common(15),
    columns=['Classification', 'Count']
)
top_classifications['Percentage'] = (top_classifications['Count'] / len(ads) * 100).round(2)

print("Top 15 Job Classifications:")
print(top_classifications.to_string(index=False))

In [ ]:
# Visualize top classifications
plt.figure(figsize=(12, 6))
plt.barh(range(10), top_classifications['Count'].head(10))
plt.yticks(range(10), top_classifications['Classification'].head(10))
plt.xlabel('Number of Job Ads')
plt.title('Top 10 Job Classifications')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Work types
work_type_counts = Counter(work_types)
work_type_df = pd.DataFrame(
    work_type_counts.most_common(),
    columns=['Work Type', 'Count']
)
work_type_df['Percentage'] = (work_type_df['Count'] / len(ads) * 100).round(2)

print("\nWork Type Distribution:")
print(work_type_df.to_string(index=False))

### Sample Ads from Different Categories

In [ ]:
# Sample diverse ads
sampled_classifications = []
sampled_ads = []

for ad in ads:
    cls = ad['metadata'].get('classification', {}).get('name', 'Unknown')
    if cls not in sampled_classifications and len(sampled_ads) < 5:
        sampled_classifications.append(cls)
        sampled_ads.append(ad)

# Display samples
for i, ad in enumerate(sampled_ads, 1):
    print(f"\n{'='*80}")
    print(f"SAMPLE AD {i}")
    print(f"{'='*80}")
    print(f"ID: {ad['id']}")
    print(f"Title: {ad['title']}")
    print(f"Classification: {ad['metadata'].get('classification', {}).get('name', 'N/A')}")
    print(f"\nAbstract:\n{ad['abstract']}")
    print(f"\nContent (first 500 chars):\n{extract_text_from_html(ad['content'])[:500]}...")

## 2. ESCO Skills Taxonomy Exploration

### Load ESCO Data

In [ ]:
# Load ESCO taxonomy
esco_df = pd.read_csv('skills_en.csv')

print(f"Total ESCO records: {len(esco_df):,}")
print(f"\nColumns: {list(esco_df.columns)}")
print(f"\nData shape: {esco_df.shape}")

In [ ]:
esco_df.head(3)

### ESCO Taxonomy Structure

In [ ]:
# Concept types
print("Concept Types:")
print(esco_df['conceptType'].value_counts())

# Skill types
if 'skillType' in esco_df.columns:
    print("\nSkill Types:")
    print(esco_df['skillType'].value_counts())

# Reuse levels
if 'reuseLevel' in esco_df.columns:
    print("\nReuse Levels:")
    print(esco_df['reuseLevel'].value_counts())

In [ ]:
# Check for missing values
print("Missing Values in Key Columns:")
missing_info = pd.DataFrame({
    'Column': ['preferredLabel', 'definition', 'altLabels', 'description'],
    'Missing Count': [
        esco_df['preferredLabel'].isna().sum(),
        esco_df['definition'].isna().sum(),
        esco_df['altLabels'].isna().sum(),
        esco_df['description'].isna().sum()
    ]
})
missing_info['Missing %'] = (missing_info['Missing Count'] / len(esco_df) * 100).round(2)
print(missing_info.to_string(index=False))

### Sample ESCO Skills

In [ ]:
# Display random sample of skills
print("Random Sample of 15 ESCO Skills:")

sample_skills = esco_df.sample(15, random_state=42)
for idx, row in sample_skills.iterrows():
    print(f"\n• {row['preferredLabel']}")
    if pd.notna(row['definition']) and row['definition']:
        print(f"  Definition: {row['definition'][:150]}..." if len(str(row['definition'])) > 150 else f"  Definition: {row['definition']}")
    if pd.notna(row['altLabels']) and row['altLabels']:
        print(f"  Alt labels: {row['altLabels']}")

## 3. Mapping Challenge Preview

### Skill Extraction Patterns

In [ ]:
# Analyze one sample ad for skill mentions
sample_for_extraction = ads[0]
sample_text = extract_text_from_html(sample_for_extraction['content'])

print("Sample Ad for Extraction Analysis:")
print("="*80)
print(f"Title: {sample_for_extraction['title']}")
print(f"\nFull Text:\n{sample_text}")

In [ ]:
# Simple pattern matching to identify potential skills
skill_patterns = [
    r'experience (?:in|with) ([^.,\n]+)',
    r'knowledge of ([^.,\n]+)',
    r'skills? in ([^.,\n]+)',
    r'proficient in ([^.,\n]+)',
    r'ability to ([^.,\n]+)',
    r'familiar with ([^.,\n]+)',
]

print("\nPotential Skill Mentions (simple pattern matching):")
print("="*80)

found_skills = []
for pattern in skill_patterns:
    matches = re.findall(pattern, sample_text.lower())
    found_skills.extend(matches)

if found_skills:
    for i, skill in enumerate(found_skills[:15], 1):
        print(f"{i}. {skill.strip()}")
else:
    print("No matches found with simple patterns.")
    print("Note: This indicates we'll need more sophisticated extraction methods.")

### Extraction Across Multiple Ads

In [ ]:
# Test pattern matching across multiple ads
all_found_skills = []

for ad in ads[:100]:  # Test on first 100 ads
    text = extract_text_from_html(ad['content'])
    for pattern in skill_patterns:
        matches = re.findall(pattern, text.lower())
        all_found_skills.extend([m.strip() for m in matches])

# Count most common extracted phrases
skill_counter = Counter(all_found_skills)

print(f"Total skill mentions extracted from first 100 ads: {len(all_found_skills)}")
print(f"Unique skill phrases: {len(skill_counter)}")
print("\nTop 20 most common skill mentions:")
print("="*80)

for skill, count in skill_counter.most_common(20):
    print(f"{count:3d}x  {skill[:80]}")

## 4. Summary

**Job Ads Dataset:**
- 50,000 job advertisements with title, abstract, and HTML content
- Diverse job classifications and work types
- Rich text content with skills, responsibilities, and requirements

**ESCO Taxonomy:**
- Comprehensive skills taxonomy with preferred labels, definitions, and alternative labels
- Multiple skill types and reuse levels